<a href="https://colab.research.google.com/github/2303A52060/High-performace-computing/blob/main/ass_8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Assignment 2: Hotspot Detection Using Timing Analysis
Scenario
A weather-simulation code shows poor performance even after
OpenMP parallelization.


Objective

 Detect hotspots (time-consuming regions)

 Focus optimization efforts on critical regions

Tasks
1. Insert timers around major code sections:

 Initialization

 Computation loop

 I/O

2. Record execution times for each section

3. Identify the section consuming maximum runtime

4. Parallelize only the hotspot region

->Expected Learning Outcome

 Learn hotspot identification

 Avoid unnecessary parallelization



In [1]:
import time
import numpy as np
from concurrent.futures import ProcessPoolExecutor
import os

alpha = 0.1

def initialize_grid(nx, ny):
    temp = np.zeros((nx, ny), dtype=np.float64)
    cx, cy = nx // 2, ny // 2
    temp[cx-5:cx+5, cy-5:cy+5] = 100.0
    velx = np.zeros_like(temp)
    vely = np.zeros_like(temp)
    return temp, velx, vely

def compute_step_serial(temp):
    nx, ny = temp.shape
    new = temp.copy()
    for i in range(1, nx-1):
        row_ip = temp[i+1]
        row_i = temp[i]
        row_im = temp[i-1]
        for j in range(1, ny-1):
            new[i, j] = row_i[j] + alpha * (
                row_ip[j] + row_im[j] + row_i[j+1] + row_i[j-1] - 4.0 * row_i[j]
            )
    return new

def compute_chunk(args):
    temp, i0, i1 = args
    temp = np.ascontiguousarray(temp)
    nx, ny = temp.shape
    chunk = np.empty((i1 - i0, ny), dtype=temp.dtype)
    for idx, i in enumerate(range(i0, i1)):
        if i == 0 or i == nx-1:
            chunk[idx, :] = temp[i, :]
            continue
        row_ip = temp[i+1]
        row_i = temp[i]
        row_im = temp[i-1]
        for j in range(0, ny):
            if j == 0 or j == ny-1:
                chunk[idx, j] = row_i[j]
            else:
                chunk[idx, j] = row_i[j] + alpha * (
                    row_ip[j] + row_im[j] + row_i[j+1] + row_i[j-1] - 4.0 * row_i[j]
                )
    return (i0, chunk)

def compute_step_parallel(temp, n_workers=4):
    nx, ny = temp.shape
    new = temp.copy()
    inner_rows = nx - 2
    # create roughly even chunks
    chunk_sizes = []
    base = inner_rows // n_workers
    rem = inner_rows % n_workers
    start = 1
    args = []
    for k in range(n_workers):
        sz = base + (1 if k < rem else 0)
        if sz == 0:
            continue
        i0 = start
        i1 = start + sz
        args.append((temp, i0, i1))
        start = i1
    with ProcessPoolExecutor(max_workers=n_workers) as ex:
        for i0, chunk in ex.map(compute_chunk, args):
            new[i0:i0+chunk.shape[0], :] = chunk
    return new

def write_output(temp, path="output.npy"):
    np.save(path, temp)

def run_sim(nx=200, ny=200, steps=10, n_workers=4):
    # Initialization
    t0 = time.perf_counter()
    temp, velx, vely = initialize_grid(nx, ny)
    t_init = time.perf_counter() - t0

    # Computation loop (serial)
    t0 = time.perf_counter()
    temp_serial = temp.copy()
    for s in range(steps):
        temp_serial = compute_step_serial(temp_serial)
    t_comp_serial = time.perf_counter() - t0

    # I/O (serial)
    t0 = time.perf_counter()
    write_output(temp_serial, "output_serial.npy")
    t_io_serial = time.perf_counter() - t0

    times = {
        "Initialization": t_init,
        "Computation (serial)": t_comp_serial,
        "I/O (serial)": t_io_serial,
    }
    hotspot = max(times, key=times.get)

    print("=== Serial run timings (seconds) ===")
    for k, v in times.items():
        print(f"{k}: {v:.6f}")
    print(f"Hotspot: {hotspot}\n")

    t0 = time.perf_counter()
    temp_par = temp.copy()
    for s in range(steps):
        temp_par = compute_step_parallel(temp_par, n_workers=n_workers)
    t_comp_parallel = time.perf_counter() - t0

    t0 = time.perf_counter()
    write_output(temp_par, "output_parallel.npy")
    t_io_parallel = time.perf_counter() - t0

    print("=== Parallelized compute timings (seconds) ===")
    print(f"Computation (parallel, workers={n_workers}): {t_comp_parallel:.6f}")
    print(f"I/O (parallel run): {t_io_parallel:.6f}")

    print("\n=== Summary ===")
    print(f"Computation speedup: {t_comp_serial / t_comp_parallel if t_comp_parallel>0 else float('inf'):.2f}x")
    print(f"Files written: {os.path.abspath('output_serial.npy')}, {os.path.abspath('output_parallel.npy')}")

if __name__ == "__main__":
    run_sim(nx=300, ny=300, steps=20, n_workers=4)

=== Serial run timings (seconds) ===
Initialization: 0.000387
Computation (serial): 1.297789
I/O (serial): 0.000430
Hotspot: Computation (serial)

=== Parallelized compute timings (seconds) ===
Computation (parallel, workers=4): 0.637360
I/O (parallel run): 0.000437

=== Summary ===
Computation speedup: 2.04x
Files written: /content/output_serial.npy, /content/output_parallel.npy


### Global Parameters

*   `alpha`: A global constant (0.1) representing the diffusion coefficient, which influences the rate of heat transfer.

### `initialize_grid(nx, ny)`

This function initializes the simulation grid. It creates a 2D NumPy array `temp` of size `nx` by `ny` to represent the temperature distribution. It sets a hot spot in the center of the grid by assigning a temperature of 100.0 to a 10x10 block. It also initializes `velx` and `vely` (velocity components) as zero arrays, although these are not used in the heat diffusion calculation in this specific script. It returns the initial `temp`, `velx`, and `vely` arrays.

### `compute_step_serial(temp)`

This function performs one step of the heat diffusion simulation using a serial (single-threaded) approach. It iterates through each inner cell of the grid (excluding boundaries) and applies the 2D heat equation (finite difference method) to update the temperature based on its neighbors' temperatures. The formula used is a discrete approximation of the Laplacian operator:

`new[i, j] = temp[i, j] + alpha * (temp[i+1, j] + temp[i-1, j] + temp[i, j+1] + temp[i, j-1] - 4 * temp[i, j])`

This function returns the updated temperature grid after one time step.

### `compute_chunk(args)`

This is a helper function designed for parallel execution. It takes a tuple `args` containing the current temperature grid `temp` and the start `i0` and end `i1` row indices for the chunk it needs to process. It computes the heat diffusion for the specified rows (chunk) of the grid, handling boundary conditions for the rows within the chunk (the very first and last rows of the *entire* grid are simply copied, while other chunk boundary rows are computed). It returns a tuple containing the starting row index `i0` and the computed `chunk` (a sub-array of the new temperature grid).

### `compute_step_parallel(temp, n_workers=4)`

This function performs one step of the heat diffusion simulation using a parallel approach with `ProcessPoolExecutor`. It divides the inner rows of the `temp` grid into roughly `n_workers` chunks. For each chunk, it calls the `compute_chunk` function using a process pool. The `ex.map` method distributes the `compute_chunk` calls across multiple processes. After all chunks are computed, it collects the results and reconstructs the `new` temperature grid. This function returns the updated temperature grid after one time step, calculated in parallel.

### `write_output(temp, path='output.npy')`

This utility function saves the final temperature grid `temp` to a binary file in NumPy's `.npy` format at the specified `path`.

### `run_sim(nx=200, ny=200, steps=10, n_workers=4)`

This is the main function that orchestrates the entire simulation and performance measurement:

1.  **Initialization**: Initializes the temperature grid and measures the time taken.
2.  **Serial Computation**: Runs the heat diffusion simulation for a specified number of `steps` using `compute_step_serial`. It measures the total computation time.
3.  **Serial I/O**: Saves the result of the serial computation to `output_serial.npy` and measures the time taken.
4.  **Performance Analysis (Serial)**: Prints the timings for initialization, serial computation, and serial I/O. It also identifies the 'hotspot' (the most time-consuming part) of the serial run.
5.  **Parallel Computation**: Runs the heat diffusion simulation for the same number of `steps` using `compute_step_parallel` with `n_workers`. It measures the total parallel computation time.
6.  **Parallel I/O**: Saves the result of the parallel computation to `output_parallel.npy` and measures the time taken.
7.  **Summary**: Prints the parallel computation and I/O timings. Finally, it calculates and prints the 'Computation speedup' (serial time / parallel time) and the absolute paths to the saved output files.